# 05 · Broadcasting: calculate across dimensions

**Level:** Intermediate · **Practice time:** 30–50 minutes

[← Previous](04_math_and_ufuncs.ipynb) · [Course guide](../README.md) · [Next →](06_statistics_and_aggregation.ipynb)

**You will learn:** Predict compatible shapes and use broadcasting for normalization and distance.

**Prerequisites:** Notebooks 00–04.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Align dimensions from the right

For broadcasting, compare shapes from right to left. Each pair must be equal or one must be 1; absent leading dimensions behave as 1. Shape (3, 2) plus (2,) works. Shape (3, 2) plus (3,) does not. Broadcasting avoids explicitly repeating the inputs, but the result still needs memory.

In [2]:
prices = np.array([[100., 200.], [150., 250.], [80., 120.]])
tax_rate = np.array([0.05, 0.10])
print("Shapes:", prices.shape, tax_rate.shape)
print("Taxed prices:\n", prices * (1 + tax_rate))
discount_by_customer = np.array([10., 20., 5.])
print("Discounted:\n", prices - discount_by_customer[:, None])
assert np.broadcast_shapes((3, 2), (3, 1)) == (3, 2)

Shapes: (3, 2) (2,)
Taxed prices:
 [[105.  220. ]
 [157.5 275. ]
 [ 84.  132. ]]
Discounted:
 [[ 90. 190.]
 [130. 230.]
 [ 75. 115.]]


**Try it:** Predict the result of broadcasting (4, 1, 3) with (2, 3). Verify with broadcast_shapes.

## Step 2 · Center columns and rows

Subtracting a column mean from every row centers each feature. `keepdims=True` preserves a length-1 axis, making the intended alignment visible. For row centering this is essential unless you insert the axis yourself.

In [3]:
data = np.array([[1., 10.], [3., 20.], [5., 30.]])
centered_columns = data - data.mean(axis=0, keepdims=True)
centered_rows = data - data.mean(axis=1, keepdims=True)
print(centered_columns)
print(centered_rows)
np.testing.assert_allclose(centered_columns.mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(centered_rows.mean(axis=1), 0, atol=1e-12)

[[ -2. -10.]
 [  0.   0.]
 [  2.  10.]]
[[ -4.5   4.5]
 [ -8.5   8.5]
 [-12.5  12.5]]


**Try it:** Remove keepdims from the row calculation. Explain why its shapes fail.

## Step 3 · Pairwise differences

A column of x values minus a row of y values creates every pairwise difference. This forms an outer operation. Here the output has shape (number of x values, number of y values). Write the shape beside each intermediate before scaling up.

In [4]:
x = np.array([1., 4., 7.])
y = np.array([2., 5.])
differences = x[:, None] - y[None, :]
print(differences)
distances = np.abs(differences)
assert distances.shape == (3, 2)
np.testing.assert_allclose(distances, [[1, 4], [2, 1], [5, 2]])

[[-1. -4.]
 [ 2. -1.]
 [ 5.  2.]]


**Try it:** Use 2D points and produce distances of shape (3, 2) by summing squared coordinate differences.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Add one offset per row to matrix.

In [5]:
matrix = np.array([[1, 2, 3], [4, 5, 6]])
offsets = np.array([10, 20])

shifted = None  # TODO: replace None with your answer

In [6]:
if shifted is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_array_equal(shifted, [[11, 12, 13], [24, 25, 26]])
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Create the 1–5 multiplication table, shape (5, 5), using broadcasting.

In [7]:
numbers = np.arange(1, 6)

times_table = None  # TODO: replace None with your answer

In [8]:
if times_table is None:
    print("Pending: complete Exercise 2 first.")
else:
    assert times_table.shape == (5, 5)
    np.testing.assert_array_equal(times_table, [[i*j for j in range(1, 6)] for i in range(1, 6)])
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

For a (4, 3) feature table, perform min-max scaling per column. Handle a constant column by returning zeros there.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Add one offset per row to matrix.

**Why it works:** Inserting a length-1 column axis turns offsets into shape (2, 1), so each row receives its own offset.

In [10]:
reference_shifted = matrix + offsets[:, None]
np.testing.assert_array_equal(reference_shifted, [[11, 12, 13], [24, 25, 26]])
print(reference_shifted)

[[11 12 13]
 [24 25 26]]


### Solution 2

Create the 1–5 multiplication table, shape (5, 5), using broadcasting.

**Why it works:** A (5, 1) column times a (1, 5) row broadcasts to all 25 products.

In [11]:
reference_times_table = numbers[:, None] * numbers[None, :]
assert reference_times_table.shape == (5, 5)
np.testing.assert_array_equal(reference_times_table, [[i*j for j in range(1, 6)] for i in range(1, 6)])
print(reference_times_table)

[[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]


## Continue learning

- [Official NumPy documentation: basics.broadcasting ](https://numpy.org/doc/stable/user/basics.broadcasting.html)

[Next →](06_statistics_and_aggregation.ipynb)